In [ ]:
import sqlite3
import tkinter as tk
from datetime import date
from decimal import Decimal, InvalidOperation
from pathlib import Path
from tkinter import messagebox, ttk


class ExpenseTrackerApp:
    def __init__(self, root):
        self.root = root
        self.root.title("Expense Tracker")
        self.root.geometry("1120x680")
        self.root.minsize(900, 560)
        self.root.configure(bg="#edf1ed")
        self.database = sqlite3.connect(Path.cwd() / "Expense Tracker.db")
        self.database.execute(
            """CREATE TABLE IF NOT EXISTS expenses (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                expense_date TEXT NOT NULL,
                payee TEXT NOT NULL,
                description TEXT NOT NULL,
                amount REAL NOT NULL CHECK (amount > 0),
                payment_method TEXT NOT NULL
            )"""
        )
        self.database.commit()
        self.selected_id = None
        self._configure_style()
        self._build_ui()
        self.refresh_expenses()
        self.root.protocol("WM_DELETE_WINDOW", self.close)

    def _configure_style(self):
        style = ttk.Style(self.root)
        style.theme_use("clam")
        style.configure("TFrame", background="#edf1ed")
        style.configure("Panel.TFrame", background="#ffffff")
        style.configure("TLabel", background="#edf1ed", foreground="#1f3029", font=("Segoe UI", 10))
        style.configure("Title.TLabel", font=("Segoe UI Semibold", 20), foreground="#183d32")
        style.configure("Muted.TLabel", foreground="#65746e")
        style.configure("TButton", font=("Segoe UI", 10), padding=(10, 7))
        style.configure("Primary.TButton", background="#187a5a", foreground="#ffffff")
        style.map("Primary.TButton", background=[("active", "#126849")])
        style.configure("Treeview", rowheight=30, font=("Segoe UI", 10), background="#ffffff", fieldbackground="#ffffff")
        style.configure("Treeview.Heading", font=("Segoe UI Semibold", 10), background="#dce8e1", foreground="#183d32")
        style.map("Treeview", background=[("selected", "#c9e7d8")], foreground=[("selected", "#183d32")])

    def _build_ui(self):
        outer = ttk.Frame(self.root, padding=20)
        outer.pack(fill="both", expand=True)
        outer.columnconfigure(1, weight=1)
        outer.rowconfigure(2, weight=1)

        header = ttk.Frame(outer)
        header.grid(row=0, column=0, columnspan=2, sticky="ew", pady=(0, 16))
        ttk.Label(header, text="Expense Tracker", style="Title.TLabel").pack(side="left")
        self.total_label = ttk.Label(header, text="Total: $0.00", style="Muted.TLabel", font=("Segoe UI Semibold", 12))
        self.total_label.pack(side="right", padx=4)

        form = ttk.Frame(outer, style="Panel.TFrame", padding=18)
        form.grid(row=1, column=0, rowspan=2, sticky="nsw", padx=(0, 16))
        ttk.Label(form, text="Expense details", font=("Segoe UI Semibold", 13), background="#ffffff").grid(row=0, column=0, columnspan=2, sticky="w", pady=(0, 14))

        self.date_var = tk.StringVar(value=date.today().isoformat())
        self.payee_var = tk.StringVar()
        self.description_var = tk.StringVar()
        self.amount_var = tk.StringVar()
        self.payment_var = tk.StringVar(value="Cash")

        fields = [
            ("Date (YYYY-MM-DD)", ttk.Entry(form, textvariable=self.date_var, width=28)),
            ("Payee", ttk.Entry(form, textvariable=self.payee_var, width=28)),
            ("Description", ttk.Entry(form, textvariable=self.description_var, width=28)),
            ("Amount", ttk.Entry(form, textvariable=self.amount_var, width=28)),
        ]
        for row, (label, widget) in enumerate(fields, start=1):
            ttk.Label(form, text=label, background="#ffffff").grid(row=row * 2 - 1, column=0, sticky="w", pady=(8, 3))
            widget.grid(row=row * 2, column=0, sticky="ew", ipady=5)

        ttk.Label(form, text="Payment method", background="#ffffff").grid(row=9, column=0, sticky="w", pady=(8, 3))
        payment = ttk.Combobox(
            form,
            textvariable=self.payment_var,
            values=("Cash", "Cheque", "Credit Card", "Debit Card", "Bank Transfer", "Other"),
            state="readonly",
            width=25,
        )
        payment.grid(row=10, column=0, sticky="ew", ipady=4)

        self.save_button = ttk.Button(form, text="Add expense", style="Primary.TButton", command=self.add_expense)
        self.save_button.grid(row=11, column=0, sticky="ew", pady=(18, 7))
        self.cancel_button = ttk.Button(form, text="Cancel edit", command=self.clear_fields, state="disabled")
        self.cancel_button.grid(row=12, column=0, sticky="ew")
        ttk.Label(form, text="Select a row to edit or delete it.", style="Muted.TLabel", background="#ffffff", wraplength=205).grid(row=13, column=0, sticky="w", pady=(18, 0))

        actions = ttk.Frame(outer)
        actions.grid(row=1, column=1, sticky="ew", pady=(0, 10))
        ttk.Button(actions, text="Edit selected", command=self.edit_selected).pack(side="left", padx=(0, 8))
        ttk.Button(actions, text="Delete selected", command=self.delete_selected).pack(side="left", padx=8)
        ttk.Button(actions, text="Delete all", command=self.delete_all).pack(side="left", padx=8)
        ttk.Button(actions, text="Read as sentence", command=self.read_selected).pack(side="left", padx=8)
        ttk.Button(actions, text="Clear fields", command=self.clear_fields).pack(side="right")

        table_frame = ttk.Frame(outer, style="Panel.TFrame", padding=8)
        table_frame.grid(row=2, column=1, sticky="nsew")
        table_frame.rowconfigure(0, weight=1)
        table_frame.columnconfigure(0, weight=1)
        columns = ("id", "date", "payee", "description", "amount", "payment")
        self.table = ttk.Treeview(table_frame, columns=columns, show="headings", selectmode="browse")
        headings = {
            "id": ("ID", 55, "center"),
            "date": ("Date", 105, "center"),
            "payee": ("Payee", 140, "w"),
            "description": ("Description", 260, "w"),
            "amount": ("Amount", 110, "e"),
            "payment": ("Payment method", 145, "w"),
        }
        for column, (label, width, anchor) in headings.items():
            self.table.heading(column, text=label)
            self.table.column(column, width=width, anchor=anchor, stretch=column in ("payee", "description", "payment"))
        vertical = ttk.Scrollbar(table_frame, orient="vertical", command=self.table.yview)
        horizontal = ttk.Scrollbar(table_frame, orient="horizontal", command=self.table.xview)
        self.table.configure(yscrollcommand=vertical.set, xscrollcommand=horizontal.set)
        self.table.grid(row=0, column=0, sticky="nsew")
        vertical.grid(row=0, column=1, sticky="ns")
        horizontal.grid(row=1, column=0, sticky="ew")
        self.table.bind("<Double-1>", lambda _event: self.edit_selected())

    def _form_values(self):
        expense_date = self.date_var.get().strip()
        payee = self.payee_var.get().strip()
        description = self.description_var.get().strip()
        amount_text = self.amount_var.get().strip()
        payment_method = self.payment_var.get().strip()
        if not all((expense_date, payee, description, amount_text, payment_method)):
            raise ValueError("Fill in the date, payee, description, amount, and payment method.")
        try:
            parsed_date = date.fromisoformat(expense_date)
        except ValueError as error:
            raise ValueError("Enter a valid date in YYYY-MM-DD format.") from error
        try:
            amount = Decimal(amount_text)
        except InvalidOperation as error:
            raise ValueError("Enter a valid amount, such as 12.50.") from error
        if not amount.is_finite() or amount <= 0:
            raise ValueError("Amount must be greater than zero.")
        return parsed_date.isoformat(), payee, description, float(amount), payment_method

    def add_expense(self):
        try:
            values = self._form_values()
            self.database.execute(
                "INSERT INTO expenses (expense_date, payee, description, amount, payment_method) VALUES (?, ?, ?, ?, ?)",
                values,
            )
            self.database.commit()
        except (ValueError, sqlite3.Error) as error:
            messagebox.showerror("Could not add expense", str(error), parent=self.root)
            return
        self.clear_fields()
        self.refresh_expenses()

    def edit_selected(self):
        selection = self.table.selection()
        if not selection:
            messagebox.showerror("No expense selected", "Select an expense to edit.", parent=self.root)
            return
        values = self.table.item(selection[0], "values")
        self.selected_id = int(values[0])
        self.date_var.set(values[1])
        self.payee_var.set(values[2])
        self.description_var.set(values[3])
        self.amount_var.set(values[4])
        self.payment_var.set(values[5])
        self.save_button.configure(text="Save changes", command=self.save_edit)
        self.cancel_button.configure(state="normal")

    def save_edit(self):
        if self.selected_id is None:
            return
        try:
            values = self._form_values()
            self.database.execute(
                "UPDATE expenses SET expense_date = ?, payee = ?, description = ?, amount = ?, payment_method = ? WHERE id = ?",
                (*values, self.selected_id),
            )
            self.database.commit()
        except (ValueError, sqlite3.Error) as error:
            messagebox.showerror("Could not save changes", str(error), parent=self.root)
            return
        self.clear_fields()
        self.refresh_expenses()

    def delete_selected(self):
        selection = self.table.selection()
        if not selection:
            messagebox.showerror("No expense selected", "Select an expense to delete.", parent=self.root)
            return
        values = self.table.item(selection[0], "values")
        if not messagebox.askyesno("Delete expense", f"Delete the expense for {values[2]}?", parent=self.root):
            return
        self.database.execute("DELETE FROM expenses WHERE id = ?", (int(values[0]),))
        self.database.commit()
        self.clear_fields()
        self.refresh_expenses()

    def delete_all(self):
        if not self.database.execute("SELECT 1 FROM expenses LIMIT 1").fetchone():
            messagebox.showinfo("No expenses", "There are no expenses to delete.", parent=self.root)
            return
        if not messagebox.askyesno("Delete all expenses", "Permanently delete every expense?", icon="warning", parent=self.root):
            return
        self.database.execute("DELETE FROM expenses")
        self.database.commit()
        self.clear_fields()
        self.refresh_expenses()

    def read_selected(self):
        selection = self.table.selection()
        if not selection:
            messagebox.showerror("No expense selected", "Select an expense to view its details.", parent=self.root)
            return
        _, expense_date, payee, description, amount, payment_method = self.table.item(selection[0], "values")
        messagebox.showinfo(
            "Expense details",
            f"You paid ${amount} to {payee} for {description} on {expense_date} via {payment_method}.",
            parent=self.root,
        )

    def clear_fields(self):
        self.selected_id = None
        self.date_var.set(date.today().isoformat())
        self.payee_var.set("")
        self.description_var.set("")
        self.amount_var.set("")
        self.payment_var.set("Cash")
        self.save_button.configure(text="Add expense", command=self.add_expense)
        self.cancel_button.configure(state="disabled")
        for item in self.table.selection():
            self.table.selection_remove(item)

    def refresh_expenses(self):
        for item in self.table.get_children():
            self.table.delete(item)
        rows = self.database.execute(
            "SELECT id, expense_date, payee, description, amount, payment_method FROM expenses ORDER BY expense_date DESC, id DESC"
        ).fetchall()
        total = Decimal("0")
        for expense_id, expense_date, payee, description, amount, payment_method in rows:
            total += Decimal(str(amount))
            self.table.insert(
                "",
                "end",
                values=(expense_id, expense_date, payee, description, f"{amount:.2f}", payment_method),
            )
        self.total_label.configure(text=f"Total: ${total:,.2f}  |  {len(rows)} expenses")

    def close(self):
        self.database.close()
        self.root.destroy()


root = tk.Tk()
app = ExpenseTrackerApp(root)
root.mainloop()
